In [61]:
from pywinauto import Desktop
dlg = Desktop(backend="uia").window(title_re=".*Microsoft Excel.*")


## Reading the files

In [62]:
def get_or_open(excel, path):
    for wb in excel.Workbooks:
        if wb.FullName.lower() == path.lower():
            return wb
    return excel.Workbooks.Open(path)

In [63]:
def get_sheet(workbook, target_name):
    target = target_name.strip().lower()

    for ws in workbook.Worksheets:
        name = ws.Name.strip().lower()
        if name == target:
            return ws

    raise Exception(f"Feuille '{target_name}' introuvable")

In [64]:
from win32com.client import DispatchEx
excel = DispatchEx("Excel.Application")
excel.Visible = False

excel.DisplayAlerts = False  # ← ICI, a
# Désactiver les updates pour la vitesse
# excel.ScreenUpdating = False
# excel.EnableEvents = False
# paths to the files
path_fichier_vl_n = r"C:\Users\aaitmoussa\Desktop\Projet Aplitec\Comptes Annuells\Notebooks\Documents\Ficheir VL n.xlsx"
path_fichier_vl_n_1 = r"C:\Users\aaitmoussa\Desktop\Projet Aplitec\Comptes Annuells\Notebooks\Documents\Fichier VL n_1.xlsx"
path_comptes_annuells = r"C:\Users\aaitmoussa\Desktop\Projet Aplitec\Comptes Annuells\Notebooks\Documents\2025.12. Sancy  .Comptes annuels.xlsm"

In [65]:
fichier_vl_n = get_or_open(excel, path_fichier_vl_n)
fichier_vl_n_1 = get_or_open(excel, path_fichier_vl_n_1)
comptes_annuells = get_or_open(excel, path_comptes_annuells)

## Extracting Balance, Inventaire (N and N-1)

#### Extracting PVMV

In [66]:
src = get_sheet(fichier_vl_n, "PVMV")
dst = get_sheet(comptes_annuells, "PVMV")


#### Clear the cells in the destination

In [67]:
dst.Cells.Clear()

True

#### copy paste

In [68]:
src.UsedRange.Copy()
dst.Range("A1").PasteSpecial()
excel.CutCopyMode = False


## Extracting Balance N and N-1

In [69]:
src_N = get_sheet(fichier_vl_n, "Balance")
src_N_1 = get_sheet(fichier_vl_n_1, "Balance")

dst_N = get_sheet(comptes_annuells, "Balance")
dst_N_1 = get_sheet(comptes_annuells, "Balance N-1")

In [70]:
dst_N.Cells.Clear()
dst_N_1.Cells.Clear()

True

#### Copy paste

In [71]:
src_N.UsedRange.Copy()
dst_N.Range("A1").PasteSpecial()

True

In [72]:
src_N_1.UsedRange.Copy()
dst_N_1.Range("A1").PasteSpecial()

True

## Extracting Inventaire N and N-1

In [73]:
src_N = get_sheet(fichier_vl_n, "Inventaire")
src_N_1 = get_sheet(fichier_vl_n_1, "Inventaire")

dst_N = get_sheet(comptes_annuells, "Inventaire")
dst_N_1 = get_sheet(comptes_annuells, "Inventaire N-1")

In [74]:
dst_N.Cells.Clear()
dst_N_1.Cells.Clear()

True

In [75]:
src_N.UsedRange.Copy()
dst_N.Range("A1").PasteSpecial()
src_N_1.UsedRange.Copy()
dst_N_1.Range("A1").PasteSpecial()

True

In [76]:
# excel.ScreenUpdating = True
# excel.EnableEvents = True

## Exécution de la macro

In [77]:
ws = comptes_annuells.Worksheets("Procédure")
macros = []
for shape in ws.Shapes:
    try:
        print(shape.Name, "->", shape.OnAction)
    except Exception:
        print(shape.Name, "(pas de macro OnAction)")

Button 13 -> '2025.12. Sancy  .Comptes annuels.xlsm'!AppliquerNumérotationContinuePiedDePage
Button 14 -> '2025.12. Sancy  .Comptes annuels.xlsm'!LierEnteteCentreMultipleOngletsAvecExclusions
Button 15 -> '2025.12. Sancy  .Comptes annuels.xlsm'!ImprimerFeuillesEnUnPDF
Button 16 -> '2025.12. Sancy  .Comptes annuels.xlsm'!ConvertirColonneATexte
Button 17 -> '2025.12. Sancy  .Comptes annuels.xlsm'!EffacerDonneesEtMiseEnFormeOnglets
TextBox 2 -> 
Button 21 -> '2025.12. Sancy  .Comptes annuels.xlsm'!RecreerCodeInventaire


In [78]:
WORKBOOK = "2025.12. Sancy  .Comptes annuels.xlsm"
macros = [
    f"'{WORKBOOK}'!ConvertirColonneATexte",
    f"'{WORKBOOK}'!AppliquerNumérotationContinuePiedDePage",
    f"'{WORKBOOK}'!LierEnteteCentreMultipleOngletsAvecExclusions",
]

In [79]:
import win32com.client
from win32com.client import Dispatch

def injecter_wrapper(excel, workbook_name):
    """Injecte un module VBA avec des wrappers qui suppriment les MsgBox."""
    
    xl = Dispatch("Excel.Application")
    wb = None
    for w in xl.Workbooks:
        if w.Name == workbook_name:
            wb = w
            break
    
    # Code VBA du wrapper
    vba_code = """
Sub RunSansMsgBox(macroName As String)
    ' Intercepter MsgBox en redéfinissant le comportement
    Application.DisplayAlerts = False
    On Error Resume Next
    Application.Run macroName
    On Error GoTo 0
    Application.DisplayAlerts = True
End Sub

Sub Wrapper_ConvertirColonneATexte()
    Application.DisplayAlerts = False
    Call ConvertirColonneATexte
    Application.DisplayAlerts = True
End Sub

Sub Wrapper_AppliquerNumérotation()
    Application.DisplayAlerts = False
    Call AppliquerNumérotationContinuePiedDePage
    Application.DisplayAlerts = True
End Sub

Sub Wrapper_LierEntete()
    Application.DisplayAlerts = False
    Call LierEnteteCentreMultipleOngletsAvecExclusions
    Application.DisplayAlerts = True
End Sub
"""
    
    # Injecter dans un nouveau module VBA
    vba_project = wb.VBProject
    
    # Supprimer le module s'il existe déjà
    for i in range(1, vba_project.VBComponents.Count + 1):
        try:
            comp = vba_project.VBComponents.Item(i)
            if comp.Name == "ModuleWrapper":
                vba_project.VBComponents.Remove(comp)
                break
        except:
            pass
    
    # Ajouter le nouveau module
    module = vba_project.VBComponents.Add(1)  # 1 = Module standard
    module.Name = "ModuleWrapper"
    module.CodeModule.AddFromString(vba_code)
    
    print("✅ Module wrapper injecté")

# Injecter une seule fois
injecter_wrapper(excel, "2025.12. Sancy  .Comptes annuels.xlsm")

# Puis appeler les wrappers au lieu des macros originales
WORKBOOK = "2025.12. Sancy  .Comptes annuels.xlsm"
macros_wrapper = [
    f"'{WORKBOOK}'!ModuleWrapper.Wrapper_ConvertirColonneATexte",
    f"'{WORKBOOK}'!ModuleWrapper.Wrapper_AppliquerNumérotation",
    f"'{WORKBOOK}'!ModuleWrapper.Wrapper_LierEntete",
]

excel.DisplayAlerts = False
for macro in macros_wrapper:
    print(f"▶ Lancement : {macro}")
    excel.Application.Run(macro)
    print(f"✅ Terminé : {macro}")
excel.DisplayAlerts = True

print("🎉 Pipeline complet !")

AttributeError: 'NoneType' object has no attribute 'VBProject'

In [ ]:
# import threading
# import time
# import win32com.client
# import pythoncom
# from pywinauto import Desktop

# def fermer_msgbox(running_flag):
#     while running_flag["active"]:
#         try:
#             dlg = Desktop(backend="uia").window(
#                 title="Microsoft Excel",
#                 class_name="#32770"
#             )
#             if dlg.exists(timeout=0.3):
#                 ok_btn = dlg.child_window(title="OK", control_type="Button")
#                 if ok_btn.exists(timeout=0.3):
#                     ok_btn.click()
#         except:
#             pass
#         time.sleep(0.2)

# def lancer_macro(macro_name):
#     pythoncom.CoInitialize()
#     try:
#         xl = win32com.client.Dispatch("Excel.Application")
#         xl.Application.Run(macro_name)
#     finally:
#         pythoncom.CoUninitialize()

# for macro in macros:
#     print(f"▶ Lancement : {macro}")
    
#     running_flag = {"active": True}
    
#     # Lancer le watcher AVANT la macro
#     t_dismiss = threading.Thread(target=fermer_msgbox, args=(running_flag,), daemon=True)
#     t_dismiss.start()
    
#     # Lancer la macro
#     t_macro = threading.Thread(target=lancer_macro, args=(macro,))
#     t_macro.start()
#     t_macro.join()
    
#     running_flag["active"] = False
#     time.sleep(0.5)
#     print(f"✅ Terminé : {macro}")

# print("🎉 Pipeline complet !")

In [ ]:
# import threading
# import time
# import win32com.client
# import pythoncom
# from pywinauto import Desktop

# # def fermer_msgbox(running_flag):
# #     while running_flag["active"]:
# #         try:
# #             dlg = Desktop(backend="uia").window(
# #                 title="Microsoft Excel",
# #                 class_name="#32770"
# #             )
# #             if dlg.exists(timeout=0.3):
# #                 ok_btn = dlg.child_window(title="OK", control_type="Button")
# #                 if ok_btn.exists(timeout=0.3):
# #                     ok_btn.click()
# #         except:
# #             pass
# #         time.sleep(0.2)

# # def lancer_macro(macro_name):
# #     pythoncom.CoInitialize()
# #     try:
# #         xl = win32com.client.Dispatch("Excel.Application")
# #         xl.Application.Run(macro_name)
# #     finally:
# #         pythoncom.CoUninitialize()

# # for macro in macros:
# #     print(f"▶ Lancement : {macro}")
    
# #     running_flag = {"active": True}
    
# #     # Lancer le watcher AVANT la macro
# #     t_dismiss = threading.Thread(target=fermer_msgbox, args=(running_flag,), daemon=True)
# #     t_dismiss.start()
    
# #     # Lancer la macro
# #     t_macro = threading.Thread(target=lancer_macro, args=(macro,))
# #     t_macro.start()
# #     t_macro.join()
    
# #     running_flag["active"] = False
# #     time.sleep(0.5)
# #     print(f"✅ Terminé : {macro}")

# # print("🎉 Pipeline complet !")
# import autoit
# import threading
# import time

# def fermer_msgbox(running_flag):
#     while running_flag["active"]:
#         try:
#             # Vérifier d'abord si la fenêtre existe
#             if autoit.win_exists("Microsoft Excel"):
#                 autoit.control_click("Microsoft Excel", "", "Button1")
#         except Exception as e:
#             pass
#         time.sleep(0.2)

# for macro in macros:
#     print(f"▶ Lancement : {macro}")
    
#     running_flag = {"active": True}
#     t_dismiss = threading.Thread(target=fermer_msgbox, args=(running_flag,), daemon=True)
#     t_dismiss.start()
    
#     excel.Application.Run(macro)
    
#     running_flag["active"] = False
#     time.sleep(0.5)
#     print(f"✅ Terminé : {macro}")

# print("🎉 Pipeline complet !")

▶ Lancement : '2025.12. Sancy  .Comptes annuels.xlsm'!ConvertirColonneATexte
✅ Terminé : '2025.12. Sancy  .Comptes annuels.xlsm'!ConvertirColonneATexte
▶ Lancement : '2025.12. Sancy  .Comptes annuels.xlsm'!AppliquerNumérotationContinuePiedDePage
✅ Terminé : '2025.12. Sancy  .Comptes annuels.xlsm'!AppliquerNumérotationContinuePiedDePage
▶ Lancement : '2025.12. Sancy  .Comptes annuels.xlsm'!LierEnteteCentreMultipleOngletsAvecExclusions
✅ Terminé : '2025.12. Sancy  .Comptes annuels.xlsm'!LierEnteteCentreMultipleOngletsAvecExclusions
🎉 Pipeline complet !
